# kaggle_arabart_finetune.ipynb
# هر بخش # ══ CELL X ══ یک سلول جداگانه در notebook است


In [ ]:
# ══ CELL 1: بررسی GPU ════════════════════════════════════════
import torch, os, sys, json, time, re
from pathlib import Path

print("=" * 50)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap = torch.cuda.get_device_capability()
    print(f"GPU: {gpu}")
    print(f"VRAM: {vram:.1f} GB")
    print(f"Capability: {cap}")
else:
    print("⚠️  CPU only — Training بسیار کند خواهد بود")
print("=" * 50)


In [ ]:

# ══ CELL 2: نصب وابستگی‌ها ═══════════════════════════════════
import subprocess

packages = [
    "transformers==4.41.2",
    "peft==0.11.1",
    "accelerate==0.31.0",
    "datasets==2.19.2",
    "rouge-score==0.1.2",
    "bert-score==0.3.13",
    "sentencepiece==0.2.0",
    "protobuf==4.25.3",
    "camel-tools==1.5.7",
    "pyarabic==0.6.15",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("✓ نصب کامل شد")


# ══ CELL 3: تنظیمات اصلی ════════════════════════════════════

In [ ]:

# ── مسیرهای Kaggle ──────────────────────────────────────────
# Dataset در Kaggle از Input می‌آید
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

# پوشه‌های خروجی
OUTPUT_DIR = KAGGLE_WORKING / "models" / "arabart"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_DIR = KAGGLE_WORKING / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset ──────────────────────────────────────────────────
# نام dataset که در Kaggle اضافه کردید را اینجا بنویسید
DATASET_NAME = "arasum-training-data"  # همان نامی که در Kaggle دادید
DATA_DIR = KAGGLE_INPUT / DATASET_NAME

# اگر dataset آپلود نشده، مسیر را دستی تنظیم کنید:
# DATA_DIR = KAGGLE_INPUT / "your-dataset-name"

print(f"Data dir: {DATA_DIR}")
print(f"Data exists: {DATA_DIR.exists()}")

if DATA_DIR.exists():
    for f in DATA_DIR.iterdir():
        size = f.stat().st_size / 1e6
        print(f"  {f.name}: {size:.1f} MB")

# ── تنظیمات مدل ─────────────────────────────────────────────
MODEL_ID = "moussaKam/AraBART"
MAX_SOURCE_LENGTH = 1024
MAX_TARGET_LENGTH = 128
SEED = 42

# ── تشخیص خودکار GPU config ─────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap = torch.cuda.get_device_capability()[0]

    if cap >= 8:        # Ampere+
        USE_FP16, USE_BF16 = False, True
    else:               # قدیمی‌تر (T4، P100)
        USE_FP16, USE_BF16 = True, False

    if vram >= 16:
        BATCH_SIZE, GRAD_ACCUM = 8, 1
    elif vram >= 12:
        BATCH_SIZE, GRAD_ACCUM = 4, 2
    else:               # P100 / T4
        BATCH_SIZE, GRAD_ACCUM = 4, 2
else:
    USE_FP16, USE_BF16 = False, False
    BATCH_SIZE, GRAD_ACCUM = 2, 4

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM

print(f"\nConfig:")
print(f"  Device: {DEVICE}")
print(f"  FP16: {USE_FP16} | BF16: {USE_BF16}")
print(f"  Batch: {BATCH_SIZE} × {GRAD_ACCUM} = {EFFECTIVE_BATCH}")



In [ ]:

# ══ CELL 4: بارگذاری دیتاست ══════════════════════════════════

def load_jsonl(path: Path, max_samples=None):
    records = []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            line = line.strip()
            if line:
                try:
                    rec = json.loads(line)
                    if rec.get("text") and rec.get("summary"):
                        records.append(rec)
                except json.JSONDecodeError:
                    pass
    return records


def load_splits(data_dir: Path):
    from datasets import Dataset

    splits = {}
    for split in ["train", "validation", "test"]:
        path = data_dir / f"{split}.jsonl"
        if path.exists():
            recs = load_jsonl(path)
            splits[split] = Dataset.from_list(recs)
            size_mb = path.stat().st_size / 1e6
            print(f"✓ {split}: {len(splits[split]):,} نمونه ({size_mb:.1f} MB)")
        else:
            print(f"⚠️  {split}: پیدا نشد در {path}")

    return splits


print("بارگذاری دیتاست ...")
splits = load_splits(DATA_DIR)

if "train" not in splits:
    raise FileNotFoundError(
        f"train.jsonl پیدا نشد در {DATA_DIR}\n"
        "مطمئن شوید Dataset را به درستی در Kaggle اضافه کردید."
    )

# نمایش چند نمونه
print("\nنمونه‌های اول:")
for i in range(2):
    ex = splits["train"][i]
    print(f"  text ({len(ex['text'].split())} کلمه): {ex['text'][:80]}...")
    print(f"  summary ({len(ex['summary'].split())} کلمه): {ex['summary'][:60]}...")
    print()



In [ ]:

# ══ CELL 5: tokenizer و مدل ══════════════════════════════════
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import set_seed

set_seed(SEED)

print(f"بارگذاری tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

print(f"بارگذاری مدل: {MODEL_ID}")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
model.to(DEVICE)

params = sum(p.numel() for p in model.parameters())
print(f"✓ مدل بارگذاری شد: {params:,} پارامتر")
print(f"  Vocab size: {tokenizer.vocab_size:,}")
print(f"  Pad token ID: {tokenizer.pad_token_id}")




In [ ]:

# ══ CELL 6: Tokenization ══════════════════════════════════════

pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

def tokenize_fn(examples):
    model_inputs = tokenizer(
        examples["text"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False,
    )
    label_enc = tokenizer(
        text_target=examples["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = [
        [t if t != pad_id else -100 for t in ids]
        for ids in label_enc["input_ids"]
    ]
    return model_inputs


print("Tokenize کردن ...")
tokenized = {}
for split_name, ds in splits.items():
    tokenized[split_name] = ds.map(
        tokenize_fn, batched=True,
        remove_columns=ds.column_names,
        desc=f"Tokenize {split_name}",
    )
    print(f"✓ {split_name}: {len(tokenized[split_name]):,} نمونه")

# نمایش نمونه
ex = tokenized["train"][0]
labels_real = [l for l in ex["labels"] if l != -100]
print(f"\nنمونه اول:")
print(f"  input_ids: {len(ex['input_ids'])} tokens")
print(f"  labels (real): {len(labels_real)} tokens")
print(f"  input: {tokenizer.decode(ex['input_ids'][:30], skip_special_tokens=True)[:80]}...")
print(f"  label: {tokenizer.decode(labels_real[:20], skip_special_tokens=True)[:60]}...")

# اندازه‌گیری truncation
for split_name, ds in tokenized.items():
    trunc = sum(1 for ex in ds if len(ex["input_ids"]) >= MAX_SOURCE_LENGTH)
    total = len(ds)
    print(f"truncation [{split_name}]: {trunc}/{total} ({100*trunc/total:.1f}%)")




In [ ]:

# ══ CELL 7: LoRA Setup ════════════════════════════════════════
from peft import LoraConfig, get_peft_model, TaskType

# شناسایی target_modules
found_modules = set()
for name, _ in model.named_modules():
    for c in ["q_proj", "v_proj", "k_proj", "out_proj"]:
        if name.endswith(c):
            found_modules.add(c)

TARGET_MODULES = sorted(list(found_modules)) or ["q_proj", "v_proj"]
print(f"Target modules: {TARGET_MODULES}")

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=TARGET_MODULES,
)

model = get_peft_model(model, lora_config)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ LoRA: {trainable:,} / {total:,} قابل‌آموزش ({100*trainable/total:.4f}%)")

if trainable == 0:
    raise RuntimeError("هیچ پارامتر قابل‌آموزشی! target_modules اشتباه است.")




In [ ]:

# ══ CELL 8: compute_metrics ══════════════════════════════════

import numpy as np
from rouge_score import rouge_scorer as rs

class ArabicTok:
    def tokenize(self, text):
        text = re.sub(r"[\u064B-\u065F\u0670\u0640]", "", text)
        return [t for t in text.split() if t]

rouge_scorer_obj = rs.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False,
    tokenizer=ArabicTok(),
)

vocab_size = tokenizer.vocab_size
_pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # کلیپ کردن برای جلوگیری از OverflowError
    predictions = np.clip(
        predictions.astype(np.int64), 0, vocab_size - 1
    ).astype(np.int32)
    labels_fixed = np.where(labels != -100, labels, _pad_id)
    labels_fixed = np.clip(
        labels_fixed.astype(np.int64), 0, vocab_size - 1
    ).astype(np.int32)

    decoded_preds, decoded_refs = [], []
    for pred_ids, label_ids in zip(predictions, labels_fixed):
        try:
            p = tokenizer.decode(
                pred_ids.tolist(), skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            ).strip()
        except Exception:
            p = ""
        try:
            r = tokenizer.decode(
                label_ids.tolist(), skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            ).strip()
        except Exception:
            r = ""
        decoded_preds.append(p)
        decoded_refs.append(r)

    r1, r2, rL = [], [], []
    for p, r in zip(decoded_preds, decoded_refs):
        if p and r:
            try:
                s = rouge_scorer_obj.score(r, p)
                r1.append(s["rouge1"].fmeasure)
                r2.append(s["rouge2"].fmeasure)
                rL.append(s["rougeL"].fmeasure)
            except Exception:
                pass

    if not r1:
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}

    return {
        "rouge1": round(sum(r1) / len(r1), 4),
        "rouge2": round(sum(r2) / len(r2), 4),
        "rougeL": round(sum(rL) / len(rL), 4),
    }

print("✓ compute_metrics آماده")





In [ ]:

# ══ CELL 9: Smoke Test (اجباری قبل از training کامل) ════════

from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

print("اجرای Smoke Test (3 step) ...")

smoke_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR / "smoke"),
    max_steps=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    eval_strategy="steps",
    eval_steps=3,
    save_strategy="steps",
    save_steps=3,
    predict_with_generate=True,
    generation_max_length=32,
    generation_num_beams=2,
    fp16=USE_FP16,
    bf16=USE_BF16,
    logging_steps=1,
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=False,
    seed=SEED,
)

smoke_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model,
    padding=True, pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

smoke_train = tokenized["train"].select(range(8))
smoke_eval = tokenized["validation"].select(range(4))

smoke_trainer = Seq2SeqTrainer(
    model=model, args=smoke_args,
    train_dataset=smoke_train,
    eval_dataset=smoke_eval,
    tokenizer=tokenizer,
    data_collator=smoke_collator,
    compute_metrics=compute_metrics,
)

smoke_result = smoke_trainer.train()
print(f"✓ Smoke Test Loss: {smoke_result.training_loss:.4f}")
print("✓ Smoke Test موفق! ادامه به Cell بعدی")




In [ ]:

# ══ CELL 10: Training کامل ═══════════════════════════════════

from transformers import EarlyStoppingCallback

# بررسی checkpoint برای resume
RESUME_FROM = None
existing_checkpoints = sorted(
    OUTPUT_DIR.glob("checkpoint-*"),
    key=lambda d: int(d.name.split("-")[-1]),
    reverse=True,
)
if existing_checkpoints:
    RESUME_FROM = str(existing_checkpoints[0])
    print(f"Resume از checkpoint: {RESUME_FROM}")
else:
    print("شروع از ابتدا")

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    seed=SEED,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    save_total_limit=3,
    logging_steps=100,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model,
    padding=True, pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"شروع training: {len(tokenized['train']):,} نمونه | {BATCH_SIZE}×{GRAD_ACCUM}={EFFECTIVE_BATCH} batch")
t0 = time.time()
result = trainer.train(resume_from_checkpoint=RESUME_FROM)
duration = time.time() - t0

print(f"\n✓ Training کامل شد!")
print(f"  زمان: {duration/3600:.2f} ساعت")
print(f"  Loss: {result.training_loss:.4f}")



In [ ]:

# ══ CELL 11: ذخیره مدل ═══════════════════════════════════════

print("ذخیره مدل ...")

# LoRA adapter
adapter_dir = OUTPUT_DIR / "lora_adapter"
adapter_dir.mkdir(exist_ok=True)
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"✓ LoRA adapter: {adapter_dir}")

# مدل merge‌شده
try:
    print("Merge کردن ...")
    merged_model = model.merge_and_unload()
    merged_dir = OUTPUT_DIR / "merged"
    merged_dir.mkdir(exist_ok=True)
    merged_model.save_pretrained(str(merged_dir))
    tokenizer.save_pretrained(str(merged_dir))
    print(f"✓ مدل merge‌شده: {merged_dir}")
except Exception as e:
    print(f"⚠️  merge ناموفق: {e}")

# metadata
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
metadata = {
    "training_time_hours": round(duration/3600, 2),
    "training_loss": result.training_loss,
    "base_model": MODEL_ID,
    "seed": SEED,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "precision": "bf16" if USE_BF16 else ("fp16" if USE_FP16 else "fp32"),
    "lora": {
        "r": 16, "alpha": 32, "dropout": 0.1,
        "target_modules": TARGET_MODULES,
        "trainable_params": trainable,
        "total_params": total,
        "trainable_pct": round(trainable/total*100, 4),
    },
    "training": {
        "epochs": 3, "lr": 5e-5,
        "batch": BATCH_SIZE, "accum": GRAD_ACCUM,
        "effective_batch": EFFECTIVE_BATCH,
    },
}

with open(OUTPUT_DIR / "training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ همه فایل‌ها در {OUTPUT_DIR} ذخیره شدند")

# نمایش محتوای output
print("\nفایل‌های خروجی:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        size = p.stat().st_size / 1e6
        print(f"  {p.relative_to(OUTPUT_DIR)}: {size:.1f} MB")






In [ ]:

# ══ CELL 12: ارزیابی نهایی (اختیاری) ════════════════════════

print("ارزیابی روی validation ...")
eval_results = trainer.evaluate()
print("\nنتایج:")
for k, v in eval_results.items():
    print(f"  {k}: {v}")

# بررسی هدف‌های پروژه
r1 = eval_results.get("eval_rouge1", 0)
r2 = eval_results.get("eval_rouge2", 0)
rL = eval_results.get("eval_rougeL", 0)

print("\n📊 بررسی اهداف پروپوزال:")
print(f"  ROUGE-1: {r1:.4f} {'✓' if r1 >= 0.45 else '✗'} (هدف ≥0.45)")
print(f"  ROUGE-2: {r2:.4f} {'✓' if r2 >= 0.25 else '✗'} (هدف ≥0.25)")
print(f"  ROUGE-L: {rL:.4f} {'✓' if rL >= 0.40 else '✗'} (هدف ≥0.40)")

# ذخیره گزارش
report = {
    "evaluation": eval_results,
    "targets": {
        "rouge1": {"target": 0.45, "achieved": r1, "pass": r1 >= 0.45},
        "rouge2": {"target": 0.25, "achieved": r2, "pass": r2 >= 0.25},
        "rougeL": {"target": 0.40, "achieved": rL, "pass": rL >= 0.40},
    },
    "metadata": metadata,
}

with open(REPORTS_DIR / "phase2_results.json", "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"\n✓ گزارش: {REPORTS_DIR / 'phase2_results.json'}")



:(